# Clinical Information Extraction Pipeline

Time estimate: **45** minutes

## Objectives

After completing this lab, you will be able to:

 - Understand the structure and components of discharge summaries
 - Build an end-to-end pipeline for extracting clinical information
 - Extract diagnoses, procedures, and medications using text processing techniques

## What you will do in this lab

In this lab, you'll create a full pipeline to extract structured medical data from unstructured clinical notes. Specifically, you'll work with discharge summaries to identify important entities like diagnoses, procedures, and medications.

You will:

- Load and inspect example discharge summaries
- Preprocess clinical text for information extraction
- Build regular expressions and simple rule-based extractors
- Extract and structure key medical information

## Overview

Discharge summaries contain vital information about a patient's hospital stay, including diagnoses, procedures performed, and prescribed medications. However, this information is often embedded in free-text format, making automated extraction a challenge.

In this lab, you will construct a pipeline that mimics real-world clinical information extraction workflows. You'll use common NLP techniques to clean, segment, and parse discharge summaries, culminating in structured outputs that can feed downstream systems like clinical decision support, reporting tools, or machine learning models.

## About the dataset

This lab uses a small set of synthetic discharge summaries modeled on real-world clinical data. The dataset is simplified and anonymized for educational purposes.

### Dataset overview

Each row in the dataset represents a discharge summary containing details such as patient diagnosis, treatment procedures, and prescribed medications. The data mimics the variability and structure commonly seen in actual clinical documentation.

### Column descriptions

1. **summary_id** - Unique identifier for each summary
2. **discharge_summary** - Full unstructured text of the discharge summary

## Setup

### Installing required libraries

The following libraries are needed to run the lab.

In [ ]:
# Install required libraries
!pip install pandas
!pip install nltk

In [ ]:
# Optional: suppress warnings
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn
warnings.filterwarnings('ignore')

### Importing required libraries

In [ ]:
# Importing libraries
import pandas as pd  # for handling data
import nltk  # for NLP tasks
import re  # for regular expressions

nltk.download('punkt')  # download tokenizer models
nltk.download('punkt_tab')  # download tokenizer models

print("Libraries loaded successfully. Ready to begin!")

## Step 1: Load and explore the discharge summaries

Start by loading the dataset and viewing a few examples to gain an understanding of its structure. Each discharge summary contains unstructured text that follows a semi-standard format with sections for diagnosis, procedures, and medications.

In [ ]:
# Load example dataset
data = {
    "summary_id": [1, 2, 3, 4],
    "discharge_summary": [
        """Patient admitted with shortness of breath and fatigue. Diagnosis: Congestive Heart Failure. Procedure: Chest X-Ray, Echocardiogram. Medications: Lasix 40mg daily, Lisinopril 10mg daily.""",
        """Admitted for chest pain radiating to left arm. Final Diagnosis: Myocardial Infarction. Procedure performed: Angioplasty with stent placement. Discharge Meds: Aspirin 81mg, Metoprolol 50mg twice daily, Atorvastatin 40mg.""",
        """Patient presented with fever and cough. Diagnosis: Community-Acquired Pneumonia. Procedure: Chest X-Ray. Medications: Azithromycin 500mg daily for 5 days, Acetaminophen as needed.""",
        """Emergency admission for severe abdominal pain. Final Diagnosis: Acute Appendicitis. Procedure performed: Laparoscopic Appendectomy. Discharge Medications: Ibuprofen 600mg every 6 hours, Amoxicillin-Clavulanate 875mg twice daily."""
    ]
}

df = pd.DataFrame(data)

# Display the first few rows
print("Dataset loaded successfully!")
print(f"\nTotal number of discharge summaries: {len(df)}")
print("\nFirst few summaries:")
df.head()

In [ ]:
# Let's examine one summary in detail
print("Example discharge summary:\n")
print(df['discharge_summary'][0])

## Step 2: Preprocess the text

Before extracting information, you need to clean and normalize the text. This step helps ensure consistency when applying our extraction patterns. You'll remove extra whitespace and normalize the text structure while preserving important clinical information.

In [ ]:
def preprocess_text(text):
    """
    Preprocess clinical text for information extraction.

    Args:
        text: Raw discharge summary text

    Returns:
        Cleaned text ready for extraction
    """
    # Remove extra whitespace and newlines
    text = re.sub(r'\s+', ' ', text)

    # Strip leading and trailing whitespace
    text = text.strip()

    return text

# Apply preprocessing to all summaries
df['cleaned_summary'] = df['discharge_summary'].apply(preprocess_text)

# Compare original and cleaned text
print("Original text:")
print(df['discharge_summary'][0][:100])
print("\nCleaned text:")
print(df['cleaned_summary'][0][:100])

## Step 3: Segment the summaries

Discharge summaries often contain multiple sentences. By segmenting the text into individual sentences, you can more easily identify which sentences contain the information you're looking for. You'll use NLTK's sentence tokenizer for this task.

In [ ]:
def segment_summary(text):
    """
    Split discharge summary into individual sentences.

    Args:
        text: Preprocessed discharge summary

    Returns:
        List of sentences
    """
    # Use NLTK's sentence tokenizer
    sentences = nltk.sent_tokenize(text)

    return sentences

# Test segmentation on the first summary
example_sentences = segment_summary(df['cleaned_summary'][0])

print("Segmented sentences from first summary:")
for i, sentence in enumerate(example_sentences, 1):
    print(f"{i}. {sentence}")

## Step 4: Extract diagnoses

Now you'll build a function to extract diagnoses from the discharge summaries. Diagnoses are typically labeled with keywords like "Diagnosis:", "Final Diagnosis:", or "Dx:". You'll use regular expressions to capture the text following these markers.

In [ ]:
def extract_diagnosis(text):
    """
    Extract diagnosis information from discharge summary.

    Args:
        text: Discharge summary text

    Returns:
        Extracted diagnosis or None if not found
    """
    # Pattern to match diagnosis keywords followed by the diagnosis text
    # The pattern looks for variations like "Diagnosis:", "Final Diagnosis:", "Dx:"
    # Case-insensitive matching with (?i)
    pattern = r'(?i)(final\s+diagnosis|diagnosis|dx):\s*([^.]+)'

    # Search for the pattern in the text
    match = re.search(pattern, text)

    if match:
        # Extract and clean the diagnosis text (group 2)
        diagnosis = match.group(2).strip()
        return diagnosis

    return None

# Apply diagnosis extraction to all summaries
df['diagnosis'] = df['cleaned_summary'].apply(extract_diagnosis)

# Display results
print("Extracted diagnoses:\n")
for idx, row in df.iterrows():
    print(f"Summary {row['summary_id']}: {row['diagnosis']}")

## Step 5: Extract procedures

Next, you'll extract procedure information. Procedures are typically marked with keywords like "Procedure:", "Procedure performed:", or "Proc:". Unlike diagnoses, there may be multiple procedures listed, so you'll capture all text following these markers until you hit another section marker or the end of a sentence.

In [ ]:
def extract_procedure(text):
    """
    Extract procedure information from discharge summary.

    Args:
        text: Discharge summary text

    Returns:
        Extracted procedure(s) or None if not found
    """
    # Pattern to match procedure keywords followed by procedure text
    # Captures variations like "Procedure:", "Procedure performed:", "Proc:"
    pattern = r'(?i)(procedure\s+performed|procedure|proc):\s*([^.]+)'

    # Search for the pattern
    match = re.search(pattern, text)

    if match:
        # Extract procedure text and clean it
        procedure = match.group(2).strip()
        return procedure

    return None

# Apply procedure extraction to all summaries
df['procedure'] = df['cleaned_summary'].apply(extract_procedure)

# Display results
print("Extracted procedures:\n")
for idx, row in df.iterrows():
    print(f"Summary {row['summary_id']}: {row['procedure']}")

## Step 6: Extract medications

Finally, you'll extract medication information. Medications are typically listed under labels like "Medications:", "Discharge Meds:", "Discharge Medications:", or "Meds:". This section often contains multiple medications with dosages, so you'll capture all text following these markers.

In [ ]:
def extract_medications(text):
    """
    Extract medication information from discharge summary.

    Args:
        text: Discharge summary text

    Returns:
        Extracted medication(s) or None if not found
    """
    # Pattern to match medication keywords followed by medication list
    # Captures variations like "Medications:", "Discharge Meds:", "Meds:"
    pattern = r'(?i)(discharge\s+medications|discharge\s+meds|medications|meds):\s*(.+?)(?=\.|$)'

    # Search for the pattern
    match = re.search(pattern, text)

    if match:
        # Extract medication text and clean it
        medications = match.group(2).strip()
        return medications

    return None

# Apply medication extraction to all summaries
df['medications'] = df['cleaned_summary'].apply(extract_medications)

# Display results
print("Extracted medications:\n")
for idx, row in df.iterrows():
    print(f"Summary {row['summary_id']}: {row['medications']}")

## Step 7: Output structured results

Now that you've extracted all the key information, let's create a clean, structured output showing the results of our extraction pipeline. This structured format can be used for reporting, analytics, or as input to downstream clinical systems.

In [ ]:
# Create a structured results dataframe with only extracted fields
results_df = df[['summary_id', 'diagnosis', 'procedure', 'medications']].copy()

# Display the structured results
print("=" * 80)
print("CLINICAL INFORMATION EXTRACTION RESULTS")
print("=" * 80)
print()

results_df

In [ ]:
# Display detailed results for each summary
for idx, row in results_df.iterrows():
    print(f"\n{'='*60}")
    print(f"SUMMARY ID: {row['summary_id']}")
    print(f"{'='*60}")
    print(f"\nDiagnosis: {row['diagnosis']}")
    print(f"\nProcedure: {row['procedure']}")
    print(f"\nMedications: {row['medications']}")

In [ ]:
# Calculate extraction statistics
total_summaries = len(results_df)
diagnosis_extracted = results_df['diagnosis'].notna().sum()
procedure_extracted = results_df['procedure'].notna().sum()
medications_extracted = results_df['medications'].notna().sum()

print("\n" + "=" * 60)
print("EXTRACTION PIPELINE STATISTICS")
print("=" * 60)
print(f"\nTotal Summaries Processed: {total_summaries}")
print(f"Diagnoses Extracted: {diagnosis_extracted} ({diagnosis_extracted/total_summaries*100:.1f}%)")
print(f"Procedures Extracted: {procedure_extracted} ({procedure_extracted/total_summaries*100:.1f}%)")
print(f"Medications Extracted: {medications_extracted} ({medications_extracted/total_summaries*100:.1f}%)")

# Exercises

Now it's your turn to implement parts of the pipeline on new samples. These exercises will help reinforce your understanding of clinical information extraction.

## Exercise 1: Extract diagnosis from a new summary

Given a new discharge summary below, use the techniques you've learned to extract the diagnosis.

In [ ]:
# New sample discharge summary
new_summary_1 = """Patient presents with persistent cough and difficulty breathing. Final Diagnosis: Chronic Obstructive Pulmonary Disease exacerbation. Treatment included nebulizer therapy."""

# Your code goes here


<details>
    <summary>Click here for a hint</summary>

Use `re.search()` to match lines containing "Diagnosis" or "Final Diagnosis" followed by a colon. Remember to use the `(?i)` flag for case-insensitive matching.

</details>

<details>
    <summary>Click here for solution</summary>

```python
# Solution for Exercise 1
# Define the pattern to match diagnosis keywords
pattern = r'(?i)(final\s+diagnosis|diagnosis):\s*([^.]+)'

# Search for the pattern in the new summary
match = re.search(pattern, new_summary_1)

if match:
    diagnosis = match.group(2).strip()
    print(f"Extracted diagnosis: {diagnosis}")
else:
    print("No diagnosis found")
```

</details>

## Exercise 2: Extract procedures from a new summary

Given the discharge summary below, extract the procedure information.

In [ ]:
# New sample discharge summary
new_summary_2 = """Patient admitted with severe joint pain. Diagnosis: Rheumatoid Arthritis. Procedure performed: MRI of bilateral knees, joint fluid aspiration. Patient tolerated procedure well."""

# Your code goes here


<details>
    <summary>Click here for a hint</summary>

Look for keywords like "Procedure" or "Procedure performed" followed by a colon. Capture everything after the colon until you hit a period.

</details>

<details>
    <summary>Click here for solution</summary>

```python
# Solution for Exercise 2
# Define the pattern to match procedure keywords
pattern = r'(?i)(procedure\s+performed|procedure):\s*([^.]+)'

# Search for the pattern
match = re.search(pattern, new_summary_2)

if match:
    procedure = match.group(2).strip()
    print(f"Extracted procedure: {procedure}")
else:
    print("No procedure found")
```

</details>

## Exercise 3: Extract medications from a new summary

Given the discharge summary below, extract all medication information including dosages.

In [ ]:
# New sample discharge summary
new_summary_3 = """Patient with Type 2 Diabetes admitted for blood sugar management. Final Diagnosis: Uncontrolled Type 2 Diabetes Mellitus. Discharge Medications: Metformin 1000mg twice daily, Glipizide 5mg before meals, Insulin glargine 20 units at bedtime."""

# Your code goes here


<details>
    <summary>Click here for a hint</summary>

Search for variations of medication keywords like "Medications", "Meds", or "Discharge Medications". Capture all text following the colon until the end of the sentence or text.

</details>

<details>
    <summary>Click here for solution</summary>

```python
# Solution for Exercise 3
# Define the pattern to match medication keywords
pattern = r'(?i)(discharge\s+medications|discharge\s+meds|medications|meds):\s*(.+?)(?=\.|$)'

# Search for the pattern
match = re.search(pattern, new_summary_3)

if match:
    medications = match.group(2).strip()
    print(f"Extracted medications: {medications}")
else:
    print("No medications found")
```

</details>

## Exercise 4: Build a complete extraction pipeline

Now, combine all three extraction functions to process a complete discharge summary and output all structured information.

In [ ]:
# New complete discharge summary
new_summary_4 = """Patient admitted via emergency department with acute onset headache and visual disturbances. Diagnosis: Migraine with aura. Procedure: CT scan of head without contrast, neurological examination. Discharge Meds: Sumatriptan 100mg as needed for migraine, Propranolol 40mg daily for prevention."""

# Your code goes here
# Extract diagnosis, procedure, and medications, then display them in a structured format


<details>
    <summary>Click here for a hint</summary>

Use the three extraction functions you've learned: `extract_diagnosis()`, `extract_procedure()`, and `extract_medications()`. Apply each function to the new summary and print the results in a clear format.

</details>

<details>
    <summary>Click here for solution</summary>

```python
# Solution for Exercise 4
# Preprocess the summary
cleaned = preprocess_text(new_summary_4)

# Extract all information
diagnosis = extract_diagnosis(cleaned)
procedure = extract_procedure(cleaned)
medications = extract_medications(cleaned)

# Display structured results
print("=" * 60)
print("EXTRACTED CLINICAL INFORMATION")
print("=" * 60)
print(f"\nDiagnosis: {diagnosis}")
print(f"\nProcedure: {procedure}")
print(f"\nMedications: {medications}")
```

</details>

# Congratulations!

You've built a complete clinical information extraction pipeline! In this lab, you explored how to preprocess discharge summaries, segment them, and extract key entities using text processing techniques. This workflow is foundational for building smarter healthcare applications that rely on unstructured data.

## Authors

Ramesh Sannareddy

Copyright © 2025 SkillUp. All rights reserved.